In [1]:
print("hello")

hello


Load 76,799 conversations
        ↓
Create official golden sample
        ↓
Inspect sample
        ↓
Label 200 examples
        ↓
Validate taxonomy
        ↓
Freeze golden set

In [2]:
import pandas as pd

amazon_conversations = pd.read_csv(
    "../data/processed/amazon_conversations.csv"
)

print("Total conversations:", len(amazon_conversations))
print("Columns:", amazon_conversations.columns.tolist())

Total conversations: 76799
Columns: ['root_tweet_id', 'thread_size', 'conversation']


In [ ]:
# Create the official golden set(frm 76799 create random 200 samples)
golden_set = amazon_conversations.sample(
    n=200,
    random_state=42
).copy()

# Add columns for manual annotation
golden_set["intent"] = ""
golden_set["annotation_notes"] = ""

print("Golden set size:", len(golden_set))

display(golden_set.head())


#thread_size is the no of tweets in 1 convo

Golden set size: 200


,root_tweet_id,thread_size,conversation,intent,annotation_notes
34891,1240036,2,CUSTOMER: Well… that’s one way to deliver a Py...,,
28089,970861,2,CUSTOMER: Amazonで注文した商品がお届け予定日過ぎてるのに届かないどころかまだ...,,
34394,1219364,4,AMAZON: @406090 Oh no! I'm sorry for the poor ...,,
43598,1559860,2,CUSTOMER: @AmazonHelp ... can you please advis...,,
43443,1554926,4,CUSTOMER: @120533 samedi 4 novembre toujours p...,,


In [6]:
golden_set.to_csv(
    "../data/golden/golden_set.csv",
    index=False
)

print("Golden set saved successfully!")
print("Location: data/golden/golden_set.csv")

Golden set saved successfully!
Location: data/golden/golden_set.csv


In [7]:
# Display the 200 golden-set conversations with a simple numbering system

for i, (_, row) in enumerate(golden_set.iterrows(), start=1):
    print("=" * 80)
    print(f"EXAMPLE {i}")
    print(f"Root Tweet ID: {row['root_tweet_id']}")
    print(f"Thread Size: {row['thread_size']}")
    print("-" * 80)
    print(row["conversation"])
    print()

EXAMPLE 1
Root Tweet ID: 1240036
Thread Size: 2
--------------------------------------------------------------------------------
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

EXAMPLE 2
Root Tweet ID: 970861
Thread Size: 2
--------------------------------------------------------------------------------
CUSTOMER: Amazonで注文した商品がお届け予定日過ぎてるのに届かないどころかまだ発送もされてない臭いんだが…(´･ω･｀)

AMAZON: @350359 ご心配をおかけしております。お届け予定日を過ぎている場合、こちらからカスタマーサービスにご連絡ください。https://t.co/J6YEizo6qC 出品者発送の商品の場合、出品者に直接ご連絡ください。https://t.co/bpqWb4Zbe4 MH

EXAMPLE 3
Root Tweet ID: 1219364
Thread Size: 4
--------------------------------------------------------------------------------
AMAZON: @406090 Oh no! I'm sorry for the poor delivery experiences. Are you noticing this trend with a specific carrier or many? ^TG

CUSTOMER: @Am

In [ ]:
#This gives us the fixed list of possible answers for our golden set.

#Later, every customer conversation will receive exactly one:


intent_labels = [
    "Delivery Issue",
    "Damaged / Wrong / Missing Item",
    "Order Management",
    "Return / Refund",
    "Payment / Billing",
    "Account / Access / Security",
    "Prime Membership",
    "Digital Content",
    "Product / Device Support",
    "Promotion / Gift Card / Credit",
    "Other / Unclear"
]

print("Number of intents:", len(intent_labels))

for i, label in enumerate(intent_labels, start=1):
    print(f"{i}. {label}")

Number of intents: 11
1. Delivery Issue
2. Damaged / Wrong / Missing Item
3. Order Management
4. Return / Refund
5. Payment / Billing
6. Account / Access / Security
7. Prime Membership
8. Digital Content
9. Product / Device Support
10. Promotion / Gift Card / Credit
11. Other / Unclear


In [ ]:
import pandas as pd

#exclude golden dataset from original dataset

# Load the full Amazon conversation pool
amazon_conversations = pd.read_csv(
    "../data/processed/amazon_conversations.csv"
)

# Load the frozen Golden Set
golden_set = pd.read_csv(
    "../data/golden/golden_set.csv"
)

# Get the conversation IDs used by the Golden Set
golden_ids = set(golden_set["root_tweet_id"])

# Remove Golden Set conversations from the development pool
dev_pool = amazon_conversations[
    ~amazon_conversations["root_tweet_id"].isin(golden_ids)
].copy()

print("Total conversations:", len(amazon_conversations))
print("Golden Set:", len(golden_set))
print("Development pool:", len(dev_pool))

print(
    "\nGolden IDs found in development pool:",
    dev_pool["root_tweet_id"].isin(golden_ids).sum()
)

Total conversations: 76799
Golden Set: 200
Development pool: 76599

Golden IDs found in development pool: 0


In [10]:
# Create a reproducible development sample

dev_sample = dev_pool.sample(
    n=10000,
    random_state=42
).copy()

print("Development sample size:", len(dev_sample))
print("Unique conversations:", dev_sample["root_tweet_id"].nunique())

Development sample size: 10000
Unique conversations: 10000
